# LeetCode 485 — Max Consecutive Ones bằng Python

**Link đề:** https://leetcode.com/problems/max-consecutive-ones/
**Độ khó:** Easy · **Chủ đề:** Array, đếm chuỗi liên tiếp (run length), cửa sổ trượt (sliding window)

```python
class Solution:
    def findMaxConsecutiveOnes(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đề: độ dài **đoạn liên tiếp toàn số 1** dài nhất trong mảng nhị phân.
2. Ôn `for`, `max`, biến đếm, `enumerate`, `str.split`, `map`, `itertools.groupby`.
3. Đi từ vét cạn mọi điểm bắt đầu `O(n²)` → **một lượt duyệt với biến `current` và `best`** `O(n)` → cách dùng chuỗi/`groupby` → cửa sổ trượt (mở đường cho bài 1004).
4. Trace từng bước, chứng minh bằng invariant, phân tích Time/Space.
5. Nhận dạng pattern "đếm run liên tiếp", làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng nhị phân `nums` (chỉ gồm 0 và 1). Trả về **số lượng số 1 liên tiếp lớn nhất**.

| `nums` | Output | Giải thích |
|---|:---:|---|
| `[1,1,0,1,1,1]` | 3 | hai số 1 đầu, hoặc ba số 1 cuối → lớn nhất 3 |
| `[1,0,1,1,0,1]` | 2 | đoạn `1,1` ở giữa |
| `[0,0,0]` | 0 | không có số 1 nào |

**Constraints:** `1 <= nums.length <= 10^5`, `nums[i]` là 0 hoặc 1.

Gợi ý của đề: hãy nghĩ về **điểm bắt đầu** và **điểm kết thúc** của mỗi "cửa sổ" toàn số 1. Một cửa sổ bắt đầu khi gặp số 1 sau số 0 (hoặc ở đầu mảng), và kết thúc khi gặp số 0 (hoặc hết mảng).

In [1]:
nums = [1, 1, 0, 1, 1, 1]
for i, v in enumerate(nums):
    print(f"chỉ số {i}: {v}")

chỉ số 0: 1
chỉ số 1: 1
chỉ số 2: 0
chỉ số 3: 1
chỉ số 4: 1
chỉ số 5: 1


Câu hỏi đặt ra trước khi code:
- Khi gặp số `0`, ta cần làm gì với "đoạn đang đếm"?
- Đoạn dài nhất có thể nằm **ở cuối mảng** — có quên cập nhật không?
- Có cần nhớ cả vị trí bắt đầu, hay chỉ cần một con số?

## 2. Nền tảng Python cần biết

### 2.1. `enumerate` — lấy cả chỉ số lẫn giá trị

`for i, v in enumerate(nums)` cho từng cặp `(chỉ số, giá trị)`. Tiện khi cần biết **vị trí** (ví dụ vị trí bắt đầu đoạn).

### 2.2. `max(a, b)` — lấy số lớn hơn

`max(3, 5) == 5`. Mẫu "cập nhật kỷ lục": `best = max(best, current)`.

In [2]:
best = 0
for current in [2, 0, 3, 1]:
    best = max(best, current)
    print(f"current = {current} -> best = {best}")

current = 2 -> best = 2
current = 0 -> best = 2
current = 3 -> best = 3
current = 1 -> best = 3


### 2.3. Biến đếm tăng/reset

Mẫu cơ bản của bài này: gặp điều kiện thì `count += 1`, gặp điều ngược lại thì `count = 0` (đặt lại về 0, gọi là **reset**).

In [3]:
count = 0
for ch in "aab-aaa":
    if ch == "a":
        count += 1
    else:
        count = 0
    print(f"{ch!r}: count = {count}")

'a': count = 1
'a': count = 2
'b': count = 0
'-': count = 0
'a': count = 1
'a': count = 2
'a': count = 3


## 3. Cách 1 — Vét cạn: thử mọi điểm bắt đầu

Với mỗi `i`, nếu `nums[i] == 1` thì đi tiếp `j = i, i+1, ...` đến khi gặp 0 hoặc hết mảng; độ dài `j - i`. Lấy max.

Trường hợp xấu (mảng toàn 1): điểm `i` phải đi `n - i` bước → tổng ~`n²/2` → `O(n²)`, quá chậm với `n = 10^5`.

In [4]:
def max_ones_bruteforce(nums: list[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        j = i
        while j < n and nums[j] == 1:
            j += 1
        best = max(best, j - i)
    return best

print(max_ones_bruteforce([1, 1, 0, 1, 1, 1]))   # 3
print(max_ones_bruteforce([1, 0, 1, 1, 0, 1]))   # 2
print(max_ones_bruteforce([0, 0, 0]))            # 0

3
2
0


Chú ý `while j < n and nums[j] == 1`: kiểm tra `j < n` **trước** để không đọc `nums[n]` (ngoài mảng). Nhờ ngắn mạch của `and`, khi `j == n` vế phải không được tính.

### 3.1. Đếm bước để thấy nút thắt

In [5]:
def count_steps_bruteforce(nums):
    n = len(nums)
    steps = 0
    for i in range(n):
        j = i
        while j < n and nums[j] == 1:
            j += 1
            steps += 1
    return steps

for n in [10, 100, 1000]:
    print(f"n = {n:>4}, toàn số 1 -> số bước ≈ {count_steps_bruteforce([1] * n)}  (n²/2 = {n * n // 2})")

n =   10, toàn số 1 -> số bước ≈ 55  (n²/2 = 50)
n =  100, toàn số 1 -> số bước ≈ 5050  (n²/2 = 5000)
n = 1000, toàn số 1 -> số bước ≈ 500500  (n²/2 = 500000)


**Nút thắt:** khi đã biết đoạn `[i, j)` toàn 1, việc bắt đầu lại từ `i + 1` và đi tới `j` một lần nữa là **lặp lại công việc đã làm**. Đoạn bắt đầu tại `i+1` chắc chắn ngắn hơn đoạn bắt đầu tại `i` (nếu cùng kết thúc ở `j`), nên chẳng cần xét.

## 4. Insight — mỗi phần tử chỉ cần xem một lần

Đi một lượt từ trái sang phải, giữ hai số:
- `current`: độ dài đoạn 1 **đang kéo dài tới vị trí hiện tại**.
- `best`: độ dài lớn nhất từng thấy.

Quy tắc:
- gặp `1` → `current += 1`, rồi `best = max(best, current)`;
- gặp `0` → `current = 0` (đoạn bị cắt đứt).

Cập nhật `best` **ngay khi** `current` tăng, nên đoạn nằm ở cuối mảng cũng được ghi nhận.

## 5. Lời giải chính — mổ xẻ từng đoạn rồi ghép

### 5.1. Khởi tạo
```python
best = current = 0
```
Gán chuỗi `a = b = 0` gán cùng giá trị cho cả hai biến.

### 5.2. Thân vòng lặp
```python
for x in nums:
    if x == 1:
        current += 1
        best = max(best, current)
    else:
        current = 0
```

### 5.3. Trả kết quả
```python
return best
```

In [6]:
def max_ones_one_pass(nums: list[int]) -> int:
    best = current = 0
    for x in nums:
        if x == 1:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best

print(max_ones_one_pass([1, 1, 0, 1, 1, 1]))   # 3
print(max_ones_one_pass([1, 0, 1, 1, 0, 1]))   # 2
print(max_ones_one_pass([0, 0, 0]))            # 0

3
2
0


## 6. TRACE từng bước — `nums = [1,1,0,1,1,1]`

| i | `x` | nhánh | `current` sau | `best` sau |
|---|---|---|---|---|
| 0 | 1 | tăng | 1 | 1 |
| 1 | 1 | tăng | 2 | 2 |
| 2 | 0 | reset | 0 | 2 |
| 3 | 1 | tăng | 1 | 2 |
| 4 | 1 | tăng | 2 | 2 |
| 5 | 1 | tăng | 3 | **3** |

Ý nghĩa: `best` chỉ thay đổi khi `current` vượt kỷ lục cũ (hàng 0, 1, 5). Hàng cuối cho thấy vì sao phải cập nhật `best` **trong** vòng lặp: đoạn dài nhất kết thúc đúng ở phần tử cuối.

In [7]:
def trace_max_ones(nums):
    best = current = 0
    for i, x in enumerate(nums):
        if x == 1:
            current += 1
            note = "tăng"
            if current > best:
                best = current
                note += " -> KỶ LỤC MỚI"
        else:
            current = 0
            note = "gặp 0 -> reset"
        print(f"i={i}, x={x}: current={current}, best={best}  ({note})")
    print("Kết quả:", best)
    return best

trace_max_ones([1, 1, 0, 1, 1, 1])
print()
trace_max_ones([1, 0, 1, 1, 0, 1])

i=0, x=1: current=1, best=1  (tăng -> KỶ LỤC MỚI)
i=1, x=1: current=2, best=2  (tăng -> KỶ LỤC MỚI)
i=2, x=0: current=0, best=2  (gặp 0 -> reset)
i=3, x=1: current=1, best=2  (tăng)
i=4, x=1: current=2, best=2  (tăng)
i=5, x=1: current=3, best=3  (tăng -> KỶ LỤC MỚI)
Kết quả: 3

i=0, x=1: current=1, best=1  (tăng -> KỶ LỤC MỚI)
i=1, x=0: current=0, best=1  (gặp 0 -> reset)
i=2, x=1: current=1, best=1  (tăng)
i=3, x=1: current=2, best=2  (tăng -> KỶ LỤC MỚI)
i=4, x=0: current=0, best=2  (gặp 0 -> reset)
i=5, x=1: current=1, best=2  (tăng)
Kết quả: 2


2

## 7. Cách 3 — Biến mảng thành chuỗi rồi `split('0')`

Ý tưởng "Pythonic": nối mảng thành chuỗi `"110111"`, tách tại mỗi `'0'` → các mảnh toàn `'1'` → độ dài mảnh dài nhất.

- `map(str, nums)` áp `str` lên từng phần tử (`1 → '1'`).
- `''.join(...)` nối thành chuỗi.
- `s.split('0')` tách chuỗi tại `'0'`; hai số 0 liền nhau tạo ra mảnh rỗng `''` (độ dài 0) — không ảnh hưởng `max`.

In [8]:
nums = [1, 1, 0, 1, 1, 1]
s = "".join(map(str, nums))
print("chuỗi:", s)
print("split('0'):", s.split("0"))
print("split với 00:", "1001".split("0"))

def max_ones_split(nums: list[int]) -> int:
    return max(len(piece) for piece in "".join(map(str, nums)).split("0"))

print(max_ones_split([1, 1, 0, 1, 1, 1]), max_ones_split([1, 0, 1, 1, 0, 1]), max_ones_split([0, 0, 0]))

chuỗi: 110111
split('0'): ['11', '111']
split với 00: ['1', '', '1']
3 2 0


Gọn, nhưng tạo chuỗi và list phụ `O(n)` bộ nhớ.

## 8. Cách 4 — `itertools.groupby`: gom các phần tử giống nhau liên tiếp

`groupby(iterable)` sinh các cặp `(key, nhóm)` cho từng **đoạn liên tiếp có cùng giá trị**. Ví dụ `[1,1,0,1,1,1]` → `(1, [1,1])`, `(0, [0])`, `(1, [1,1,1])`. Đây chính là **run-length encoding** (mã hóa độ dài chạy).

Nhóm là một iterator (chỉ duyệt được một lần) → đổi sang `list` để xem, hoặc đếm bằng `sum(1 for _ in g)`.

In [9]:
from itertools import groupby

for key, group in groupby([1, 1, 0, 1, 1, 1]):
    print(key, list(group))

def max_ones_groupby(nums: list[int]) -> int:
    best = 0
    for key, group in groupby(nums):
        if key == 1:
            best = max(best, sum(1 for _ in group))
    return best

print(max_ones_groupby([1, 1, 0, 1, 1, 1]), max_ones_groupby([0, 0, 0]))

1 [1, 1]
0 [0]
1 [1, 1, 1]
3 0


### 8.1. Cách 5 — Cửa sổ trượt với hai chỉ số `left`, `right`

Giữ cửa sổ `[left, right]` **toàn số 1**. Khi `nums[right] == 0` thì cửa sổ "vỡ": dời `left = right + 1`. Độ dài cửa sổ là `right - left + 1`.

Với bài này, cách này tương đương cách 2 (`current = right - left + 1`), nhưng là **khuôn mẫu** cho biến thể "được lật tối đa `k` số 0" (LeetCode 1004) ở mục biến thể.

In [10]:
def max_ones_window(nums: list[int]) -> int:
    best = 0
    left = 0
    for right, x in enumerate(nums):
        if x == 0:
            left = right + 1               # cửa sổ mới bắt đầu sau số 0
        else:
            best = max(best, right - left + 1)
    return best

print(max_ones_window([1, 1, 0, 1, 1, 1]), max_ones_window([1, 0, 1, 1, 0, 1]), max_ones_window([0]))

3 2 0


## 9. Code nộp LeetCode (Python 3)

In [11]:
class Solution:
    def findMaxConsecutiveOnes(self, nums: list[int]) -> int:
        best = current = 0
        for x in nums:
            if x == 1:
                current += 1
                if current > best:
                    best = current
            else:
                current = 0
        return best

sol = Solution()
print(sol.findMaxConsecutiveOnes([1, 1, 0, 1, 1, 1]))   # 3
print(sol.findMaxConsecutiveOnes([1, 0, 1, 1, 0, 1]))   # 2

3
2


(`if current > best: best = current` thay cho `max(...)` chỉ để tránh gọi hàm mỗi vòng — nhanh hơn chút xíu trong Python, ý nghĩa như nhau.)

## 10. Vì sao đúng? (invariant)

Sau khi xử lý phần tử thứ `i`:
- **(I1)** `current` = độ dài đoạn 1 liên tiếp **kết thúc tại `i`** (bằng 0 nếu `nums[i] = 0`).
- **(I2)** `best` = max độ dài các đoạn 1 liên tiếp **nằm trong `nums[0..i]`**.

Chứng minh bằng quy nạp:
- Nếu `nums[i] = 1`: đoạn kết thúc tại `i` = đoạn kết thúc tại `i-1` nối thêm `i` → `current + 1` ✔ (I1). Mọi đoạn trong `nums[0..i]` hoặc nằm trong `nums[0..i-1]` (đã có trong `best` cũ), hoặc kết thúc tại `i` (dài nhất là `current` mới) → `max(best, current)` ✔ (I2).
- Nếu `nums[i] = 0`: không đoạn 1 nào kết thúc tại `i` → `current = 0` ✔; `best` giữ nguyên ✔.

Khi kết thúc, (I2) với `i = n-1` chính là đáp án.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Vét cạn mọi điểm bắt đầu | `O(n²)` | `O(1)` | TLE với mảng toàn 1 |
| 2. Một lượt `current/best` | `O(n)` | `O(1)` | **chuẩn** |
| 3. `join` + `split('0')` | `O(n)` | `O(n)` | gọn, tốn bộ nhớ |
| 4. `groupby` | `O(n)` | `O(1)`* | run-length |
| 5. Cửa sổ trượt | `O(n)` | `O(1)` | khuôn cho bài 1004 |

\* `groupby` không tạo list; nhưng nếu đổi nhóm sang `list` thì tốn bộ nhớ bằng độ dài nhóm.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", [1, 1, 0, 1, 1, 1], 3),
    ("đề 2", [1, 0, 1, 1, 0, 1], 2),
    ("toàn 0", [0, 0, 0], 0),
    ("toàn 1", [1] * 7, 7),
    ("một phần tử 1", [1], 1),
    ("một phần tử 0", [0], 0),
    ("dài nhất ở đầu", [1, 1, 1, 0, 1], 3),
    ("dài nhất ở cuối", [0, 1, 0, 1, 1], 2),
    ("xen kẽ", [1, 0] * 5, 1),
]
sol = Solution()
for name, nums, expected in cases:
    check(name, sol.findMaxConsecutiveOnes(nums), expected)

PASS  đề 1: 3
PASS  đề 2: 2
PASS  toàn 0: 0
PASS  toàn 1: 7
PASS  một phần tử 1: 1
PASS  một phần tử 0: 0
PASS  dài nhất ở đầu: 3
PASS  dài nhất ở cuối: 2
PASS  xen kẽ: 1


In [13]:
import random
random.seed(485)
methods = [max_ones_bruteforce, max_ones_one_pass, max_ones_split, max_ones_groupby, max_ones_window]
for _ in range(2000):
    nums = [random.randint(0, 1) for _ in range(random.randint(1, 20))]
    results = {f(nums) for f in methods}
    assert len(results) == 1, (nums, results)
print("2000 mảng ngẫu nhiên: 5 cách khớp nhau")

big = [1] * 100_000
print("mảng toàn 1 dài 10^5:", Solution().findMaxConsecutiveOnes(big))

2000 mảng ngẫu nhiên: 5 cách khớp nhau
mảng toàn 1 dài 10^5: 100000


`100_000` — Python cho phép dùng dấu gạch dưới trong số để dễ đọc; giá trị vẫn là `100000`.

## 13. Lỗi người mới hay mắc

### Lỗi 1: chỉ cập nhật `best` khi gặp số 0
Nếu đoạn dài nhất ở **cuối mảng**, không có số 0 nào sau nó → không bao giờ được ghi nhận.

In [14]:
def wrong_update_on_zero(nums):
    best = current = 0
    for x in nums:
        if x == 1:
            current += 1
        else:
            best = max(best, current)   # chỉ cập nhật ở đây
            current = 0
    return best                         # SAI: quên đoạn cuối

print("sai :", wrong_update_on_zero([1, 1, 0, 1, 1, 1]))
print("đúng:", max_ones_one_pass([1, 1, 0, 1, 1, 1]))

sai : 2
đúng: 3


Sửa: hoặc cập nhật `best` mỗi khi `current` tăng (như lời giải chính), hoặc thêm `return max(best, current)` sau vòng lặp.

### Lỗi 2: quên reset `current` khi gặp 0
`current` khi đó đếm **tổng số 1** chứ không phải số 1 liên tiếp.

In [15]:
def wrong_no_reset(nums):
    best = current = 0
    for x in nums:
        if x == 1:
            current += 1
            best = max(best, current)
    return best

print("sai :", wrong_no_reset([1, 0, 1, 1, 0, 1]))
print("đúng:", max_ones_one_pass([1, 0, 1, 1, 0, 1]))

sai : 4
đúng: 2


### Lỗi 3: khởi tạo `best = 1` (giả định luôn có số 1)
Mảng toàn 0 sẽ trả 1 thay vì 0. Luôn khởi tạo `best = 0`.

### Lỗi 4: dùng `split('0')` trên list thay vì chuỗi
`list` không có phương thức `split` → `AttributeError`. Phải `''.join(map(str, nums))` trước.

In [16]:
try:
    [1, 0, 1].split(0)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'list' object has no attribute 'split'


### Lỗi 5: dùng `str(nums)` để tạo chuỗi
`str([1, 1, 0])` là `'[1, 1, 0]'` (có ngoặc, dấu phẩy, khoảng trắng) — tách theo `'0'` sẽ đếm cả ký tự rác.

In [17]:
print(repr(str([1, 1, 0, 1])))
print("sai :", max(len(p) for p in str([1, 1, 0, 1]).split("0")))
print("đúng:", max_ones_split([1, 1, 0, 1]))

'[1, 1, 0, 1]'
sai : 7
đúng: 2


## 14. Mẹo nhận dạng pattern

- "Đoạn **liên tiếp** dài nhất thỏa điều kiện đơn giản (từng phần tử)" → **một biến đếm + reset** (`current`), một biến kỷ lục (`best`).
- Điều kiện phức tạp hơn ("được phép vi phạm tối đa `k` lần") → **cửa sổ trượt** hai con trỏ.
- Cần thông tin từng "khúc" (giá trị + độ dài) → **`groupby`** / run-length encoding.

Bài cùng họ: 487 Max Consecutive Ones II (Premium, lật tối đa 1 số 0), 1004 Max Consecutive Ones III, 1446 Consecutive Characters, 1869 Longer Contiguous Segments of Ones than Zeros, 674 Longest Continuous Increasing Subsequence, 3 Longest Substring Without Repeating Characters.

## 15. Biến thể

### 15.1. Được lật tối đa `k` số 0 thành 1 (LeetCode 1004)

Cửa sổ trượt: mở rộng `right`; đếm số 0 trong cửa sổ; khi số 0 vượt `k` thì **thu hẹp** `left` cho đến khi hợp lệ trở lại. Mỗi chỉ số vào/ra cửa sổ tối đa một lần → `O(n)`.

In [18]:
def longest_ones_k(nums: list[int], k: int) -> int:
    left = zeros = best = 0
    for right, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > k:                 # cửa sổ không hợp lệ -> thu hẹp
            if nums[left] == 0:
                zeros -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

print(longest_ones_k([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2))                       # 6
print(longest_ones_k([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3))  # 10
print(longest_ones_k([1, 1, 0, 1, 1, 1], 0))                                     # 3 = bài gốc

6
10
3


Với `k = 0`, biến thể quay về đúng bài 485.

### 15.2. Ký tự giống nhau liên tiếp dài nhất (LeetCode 1446)
Thay điều kiện "bằng 1" bằng "bằng ký tự trước đó".

In [19]:
def max_power(s: str) -> int:
    best = current = 1
    for i in range(1, len(s)):
        current = current + 1 if s[i] == s[i - 1] else 1
        best = max(best, current)
    return best

print(max_power("leetcode"), max_power("abbcccddddeeeeedcba"))   # 2 5

2 5


`a if cond else b` là **biểu thức điều kiện** (toán tử ba ngôi): trả `a` nếu `cond` đúng, ngược lại `b`.

## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Tính tay
Lập bảng `current`, `best` cho `nums = [0, 1, 1, 0, 1, 1, 1, 1, 0]`.

### Bài 2 — Điền chỗ trống

In [20]:
def ex2_max_ones(nums: list[int]) -> int:
    # TODO 1: best = current = 0
    # TODO 2: duyệt x; gặp 1 thì tăng current và cập nhật best
    # TODO 3: gặp 0 thì reset current
    pass

# assert ex2_max_ones([1, 1, 0, 1, 1, 1]) == 3
# assert ex2_max_ones([0]) == 0

### Bài 3 — Trả về cả **vị trí bắt đầu** của đoạn dài nhất (đoạn đầu tiên nếu hòa)
Ví dụ `[1, 1, 0, 1, 1, 1]` → `(3, 3)` (bắt đầu tại chỉ số 3, dài 3). Nếu không có số 1 → `(-1, 0)`.

In [21]:
def ex3_longest_run_with_start(nums: list[int]) -> tuple[int, int]:
    # TODO
    pass

# assert ex3_longest_run_with_start([1, 1, 0, 1, 1, 1]) == (3, 3)
# assert ex3_longest_run_with_start([1, 1, 0, 1, 1]) == (0, 2)
# assert ex3_longest_run_with_start([0, 0]) == (-1, 0)

### Bài 4 — Longer Contiguous Segments (LeetCode 1869)
Cho chuỗi nhị phân `s`, trả `True` nếu đoạn `'1'` liên tiếp dài nhất **dài hơn hẳn** đoạn `'0'` liên tiếp dài nhất. Ví dụ `"1101"` → `True`, `"111000"` → `False`, `"110100010"` → `False`.

In [22]:
def ex4_check_zero_ones(s: str) -> bool:
    # TODO
    pass

# assert ex4_check_zero_ones("1101") is True
# assert ex4_check_zero_ones("111000") is False
# assert ex4_check_zero_ones("110100010") is False

### Bài 5 — Mini challenge: lật tối đa **một** số 0 (ý tưởng bài 487)
Không dùng cửa sổ tổng quát; hãy giữ hai biến: `cur` (số 1 liên tiếp kết thúc tại `i`, không lật) và `flip` (độ dài đoạn kết thúc tại `i` đã dùng đúng ≤ 1 lần lật). Ví dụ `[1,0,1,1,0]` → `4`, `[1,0,1,1,0,1]` → `4`, `[0]` → `1`.

In [23]:
def ex5_max_ones_one_flip(nums: list[int]) -> int:
    # TODO
    pass

# assert ex5_max_ones_one_flip([1, 0, 1, 1, 0]) == 4
# assert ex5_max_ones_one_flip([1, 0, 1, 1, 0, 1]) == 4
# assert ex5_max_ones_one_flip([0]) == 1
# assert ex5_max_ones_one_flip([1, 1, 1]) == 3

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: `current` reset ở mỗi số 0. Mức 2: đoạn thứ hai dài 4. Mức 3: `best` cuối = 4.

**Bài 2** — Mức 1: chép khung mục 5. Mức 2: `current += 1` rồi `best = max(...)`. Mức 3: `else: current = 0`.

**Bài 3** — Mức 1: nhớ `start` của đoạn hiện tại (vị trí ngay sau số 0 gần nhất). Mức 2: khi `current > best` thì lưu `best_start = start`. Mức 3: dùng `>` (không phải `>=`) để giữ đoạn **đầu tiên** khi hòa.

**Bài 4** — Mức 1: viết hàm `longest(s, ch)`. Mức 2: `groupby(s)` cho từng khúc. Mức 3: `return longest(s, '1') > longest(s, '0')`.

**Bài 5** — Mức 1: gặp `1`: `cur += 1`, `flip += 1`. Mức 2: gặp `0`: `flip = cur + 1` (lật số 0 này, nối với đoạn 1 ngay trước), `cur = 0`. Mức 3: `best = max(best, flip)` mỗi bước.

## 17. Đáp án tham khảo

**Bài 1:** `current`: 0,1,2,0,1,2,3,4,0 — `best`: 0,1,2,2,2,2,3,4,4 → **4**.

In [24]:
def ans2_max_ones(nums):
    best = current = 0
    for x in nums:
        if x == 1:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best

def ans3_longest_run_with_start(nums):
    best, best_start = 0, -1
    current, start = 0, 0
    for i, x in enumerate(nums):
        if x == 1:
            if current == 0:
                start = i
            current += 1
            if current > best:
                best, best_start = current, start
        else:
            current = 0
    return (best_start, best)

def ans4_check_zero_ones(s):
    def longest(ch):
        return max((sum(1 for _ in g) for key, g in groupby(s) if key == ch), default=0)
    return longest("1") > longest("0")

def ans5_max_ones_one_flip(nums):
    best = cur = flip = 0
    for x in nums:
        if x == 1:
            cur += 1
            flip += 1
        else:
            flip = cur + 1
            cur = 0
        best = max(best, flip)
    return best

assert ans2_max_ones([0, 1, 1, 0, 1, 1, 1, 1, 0]) == 4        # bài 1
assert ans2_max_ones([1, 1, 0, 1, 1, 1]) == 3 and ans2_max_ones([0]) == 0
assert ans3_longest_run_with_start([1, 1, 0, 1, 1, 1]) == (3, 3)
assert ans3_longest_run_with_start([1, 1, 0, 1, 1]) == (0, 2)
assert ans3_longest_run_with_start([0, 0]) == (-1, 0)
assert ans4_check_zero_ones("1101") is True
assert ans4_check_zero_ones("111000") is False
assert ans4_check_zero_ones("110100010") is False
assert ans5_max_ones_one_flip([1, 0, 1, 1, 0]) == 4
assert ans5_max_ones_one_flip([1, 0, 1, 1, 0, 1]) == 4
assert ans5_max_ones_one_flip([0]) == 1
assert ans5_max_ones_one_flip([1, 1, 1]) == 3
# đối chiếu bài 5 với cửa sổ trượt k = 1 trên dữ liệu ngẫu nhiên
for _ in range(500):
    arr = [random.randint(0, 1) for _ in range(random.randint(1, 15))]
    assert ans5_max_ones_one_flip(arr) == longest_ones_k(arr, 1), arr
print("Đáp án bài 1–5: PASS (bài 5 khớp cửa sổ trượt k = 1 trên 500 mảng)")

Đáp án bài 1–5: PASS (bài 5 khớp cửa sổ trượt k = 1 trên 500 mảng)


`max(iterable, default=0)`: nếu iterable rỗng (ví dụ chuỗi không có ký tự `'0'`) thì trả `default` thay vì báo lỗi `ValueError`.

## 18. Template nhớ nhanh + checklist submit

```python
best = current = 0
for x in nums:
    if x == 1:              # điều kiện "còn nối dài được"
        current += 1
        best = max(best, current)
    else:
        current = 0         # đứt đoạn -> reset
return best
```

**Checklist trước khi submit**
- [ ] `best` được cập nhật **trong** vòng lặp (không chỉ khi gặp 0)?
- [ ] Có reset `current = 0` khi gặp 0?
- [ ] Khởi tạo `best = 0` (mảng toàn 0 → 0)?
- [ ] Thử ca: đoạn dài nhất ở cuối, mảng một phần tử, toàn 1?
- [ ] Nói được invariant của `current` và `best`?